# 06 - Replica y exportacion de documentos MGCECDL

Este cuaderno toma como base el flujo del `05_Analisis_Circuito_MGCECDL.ipynb` y genera, con **todos los datos disponibles**, los cuatro documentos equivalentes a los producidos previamente en inferencia:

1. Top-20 por evento con valores originales y relevancia del modelo.
2. Agregado por vano.
3. Agregado por circuito.
4. Resumen por circuito y nivel de criticidad.

Cada apartado muestra una vista de control en el cuaderno y guarda su CSV en `data/results_mgcecdl/`. El Top-20 se calcula con Kernel SHAP sobre el clasificador MGCECDL; los agrupamientos posteriores reutilizan ese Top-20 por evento y agregan relevancia con Borda ponderado por la magnitud SHAP.

In [1]:
import importlib
import json
import io
import os
import shutil
import subprocess
import sys
import warnings
from contextlib import redirect_stdout
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Configuracion minima para que el cuaderno funcione localmente o en Kaggle.
REPO_URL = "https://github.com/jclugor/chec-local-uiti-vano-interpreter.git"
REPO_NAME = "chec-local-uiti-vano-interpreter"


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "chec_impacto").exists():
            return candidate
    working_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else cwd
    clone_dir = working_root / REPO_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )


def ensure_lfs_data(project_root):
    data_path = project_root / "data" / "Indicadores_vano_v3.csv"
    if shutil.which("git-lfs") and (project_root / ".git").exists():
        subprocess.run(["git", "lfs", "install", "--local"], cwd=project_root, check=False)
        subprocess.run(["git", "lfs", "pull"], cwd=project_root, check=True)
    if data_path.exists() and data_path.stat().st_size < 1024:
        head = data_path.read_text(errors="ignore")[:120]
        if "git-lfs" in head:
            raise RuntimeError(
                "Indicadores_vano_v3.csv quedo como puntero Git LFS. "
                "Descarga el archivo LFS antes de continuar."
            )


# Resolver proyecto, asegurar dependencias/datos y cargar el paquete local.
PROJECT_ROOT = resolve_project_root()
install_project_requirements(PROJECT_ROOT)
ensure_lfs_data(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

for module_name in [
    "chec_impacto.models.mgcecdl",
    "chec_impacto.training.mgcecdl",
    "chec_impacto.training",
    "chec_impacto.interpretability.circuit_analysis",
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

# Todas las rutas relativas del cuaderno quedan ancladas al repo.
os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

Updated Git hooks.
Git LFS initialized.
PROJECT_ROOT: /Users/diego/Documents/CHEC/chec_impacto
DATA_DIR: /Users/diego/Documents/CHEC/chec_impacto/data


## Parametros

Esta celda deja el flujo en modo produccion: usa todos los eventos procesados, no filtra circuitos y define la carpeta de salida `data/results_mgcecdl/`. Los parametros SHAP controlan el costo computacional del Top-20 por evento.

In [2]:
# Flujo de produccion: no se muestrea y no se filtran circuitos.
MODO_AUDITORIA          = False
MAX_EVENTOS_AUDITORIA   = None
CIRCUITOS_AUDITORIA     = []  # Ejemplo para depurar: ["DON23L13"]. Vacio = todos los circuitos.

# Parametros del Top-20 por evento. Subir nsamples mejora fidelidad SHAP, pero aumenta el tiempo.
TOP_K_VARS              = 20
RANDOM_STATE            = 42
SHAP_BACKGROUND_SIZE    = 40
SHAP_NSAMPLES           = 80
SHAP_BATCH_SIZE         = 512
SHAP_RANDOM_STATE       = 42
FILTRO_UITI_MAX         = None
VENTANA_CLIMATICA_HORAS = 12

DATASET_PATH             = DATA_DIR / "Indicadores_vano_v3.csv"
VARIABLES_SELECCION_PATH = DATA_DIR / "Variables_seleccion.xlsx"
RESULTS_DIR              = DATA_DIR / "results_mgcecdl"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = {
    "top20_eventos": RESULTS_DIR / "Indicadores_vano_v3_top20_mgcecdl_clasificacion.csv",
    "agrupado_por_vano": RESULTS_DIR / "Indicadores_vano_v3_agrupado_por_vano.csv",
    "agrupado_por_circuito": RESULTS_DIR / "Indicadores_vano_v3_agrupado_por_circuito.csv",
    "resumen_criticidad": RESULTS_DIR / "Indicadores_vano_v3_resumen_circuito_por_nivel_criticidad.csv",
}


def _modelo_mas_reciente(model_dir: Path, base_name: str) -> Path:
    stem = Path(base_name).stem
    suffix = Path(base_name).suffix
    candidates = sorted(model_dir.glob(f"{stem}*{suffix}"))
    if not candidates:
        raise FileNotFoundError(f"No encontrado: {base_name} en {model_dir}")
    selected = candidates[-1]
    if len(candidates) > 1:
        print(f"  {len(candidates)} versiones de {base_name}. Usando: {selected.name}")
    return selected


def _json_default(value):
    # Convierte tipos frecuentes de pandas/numpy a valores compatibles con JSON.
    if pd.isna(value):
        return None
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).isoformat()
    if isinstance(value, pd.Timedelta):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    return str(value)


def _serializar_columnas_dict(df: pd.DataFrame) -> pd.DataFrame:
    # Convierte columnas con diccionarios a JSON para que el CSV sea estable.
    out = df.copy()
    for col in out.columns:
        if out[col].map(lambda value: isinstance(value, dict)).any():
            out[col] = out[col].map(
                lambda value: json.dumps(value, ensure_ascii=False, default=_json_default)
                if isinstance(value, dict)
                else value
            )
    return out


def guardar_resultado(df: pd.DataFrame, output_path: Path, nombre: str) -> None:
    # Guarda un apartado como CSV con BOM UTF-8 para compatibilidad con Excel.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    _serializar_columnas_dict(df).to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Guardado {nombre}: {output_path} | filas={len(df):,} | columnas={df.shape[1]:,}")


MODEL_PATH = _modelo_mas_reciente(DATA_DIR / "models", "mgcecdl_classifier_best.zip")
for _p in [DATASET_PATH, VARIABLES_SELECCION_PATH, MODEL_PATH]:
    if not _p.exists():
        raise FileNotFoundError(f"No encontrado: {_p}")

print("Modelo MGCECDL:", MODEL_PATH.name)
print("Modo auditoria:", MODO_AUDITORIA)
print("Carpeta de salida:", RESULTS_DIR)


Modelo MGCECDL: mgcecdl_classifier_best.zip
Modo auditoria: False
Carpeta de salida: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl


## 1. Carga del dataset y escalado MGCECDL

Se procesa el dataset completo con la misma preparacion usada en entrenamiento. Para MGCECDL, las entradas del modelo se escalan con el `feature_scaler`, pero las tablas exportadas conservan los valores originales del dataset base. En los agrupamientos, `UITI` se calcula como suma.

In [3]:
from chec_impacto.data import preparar_splits_estratificados, procesar_dataset_completo
from chec_impacto.training import escalar_features_minmax_mgcecdl, resolve_training_device

# Seleccionar el mejor dispositivo disponible para inferencia MGCECDL.
DEVICE = resolve_training_device("auto")
print(f"Usando device: {DEVICE}")

# La preparacion imprime muchos detalles; se silencian para que la salida sea de control.
with redirect_stdout(io.StringIO()):
    datos = procesar_dataset_completo(
        path_clima=DATASET_PATH,
        path_variables_seleccion=VARIABLES_SELECCION_PATH,
        use_sampling=False,
        min_samples_per_codigo=5,
        target="UITI_VANO",
        filtro_uiti_max=FILTRO_UITI_MAX,
        ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
    )

X_full_raw = np.asarray(datos["X"], dtype=np.float32)
features = list(datos["features"])
base_full = datos["df_original_copy"].copy().reset_index(drop=True)

# Reutilizar el escalado min-max esperado por el clasificador MGCECDL.
splits_clf = escalar_features_minmax_mgcecdl(
    preparar_splits_estratificados(
        X_full_raw,
        datos["y"],
        modo="clasificacion",
        random_state=RANDOM_STATE,
    )
)
feature_scaler = splits_clf["feature_scaler"]
X_full = feature_scaler.transform(X_full_raw).astype(np.float32)

print(f"Eventos procesados : {len(base_full):,}")
print(f"Vanos unicos       : {base_full['FID_VANO'].nunique():,}")
print(f"Circuitos          : {base_full['CIRCUITO'].nunique():,}")
print(f"Features modelo    : {len(features):,}")

/Users/diego/Documents/CHEC/chec_impacto/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usando device: mps
Dataset original: X=(159470, 70), y=(159470, 1)
Splits generados -> Train: (102060, 70), Valid: (25516, 70), Test: (31894, 70)
Modo objetivo: clasificacion

Distribución de clases para estratificación:
Original: [39868 39867 39867 39868]
Train:    [25515 25515 25515 25515]
Valid:    [6379 6379 6379 6379]
Test:     [7974 7973 7973 7974]
Eventos procesados : 159,470
Vanos unicos       : 27,390
Circuitos          : 208
Features modelo    : 70


## 2. Seleccion de datos de trabajo

Esta version usa todos los datos disponibles despues del procesamiento. La lista `CIRCUITOS_AUDITORIA` queda como control opcional, pero por defecto esta vacia para no excluir circuitos.

In [4]:
# Partimos del dataset completo procesado; solo se filtra si se define CIRCUITOS_AUDITORIA.
base_filtrada = base_full.copy()
X_filtrado = X_full

if CIRCUITOS_AUDITORIA:
    mascara_circuitos = base_filtrada["CIRCUITO"].astype(str).str.strip().isin(CIRCUITOS_AUDITORIA)
    if not mascara_circuitos.any():
        raise ValueError(f"Sin eventos para CIRCUITOS_AUDITORIA={CIRCUITOS_AUDITORIA}")
    X_filtrado = X_filtrado[mascara_circuitos.to_numpy()]
    base_filtrada = base_filtrada[mascara_circuitos].copy().reset_index(drop=True)

# En produccion MODO_AUDITORIA=False, por lo que pasan todos los eventos disponibles.
if MODO_AUDITORIA:
    if MAX_EVENTOS_AUDITORIA is None:
        raise ValueError("MAX_EVENTOS_AUDITORIA debe tener valor cuando MODO_AUDITORIA=True.")
    n_auditoria = min(MAX_EVENTOS_AUDITORIA, len(base_filtrada))
    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.sort(rng.choice(len(base_filtrada), size=n_auditoria, replace=False))
    X = X_filtrado[indices]
    base = base_filtrada.iloc[indices].copy().reset_index(drop=True)
else:
    X = X_filtrado
    base = base_filtrada.copy().reset_index(drop=True)

print(f"Eventos en trabajo : {len(base):,}")
print(f"Vanos en trabajo   : {base['FID_VANO'].nunique():,}")
print(f"Circuitos trabajo  : {base['CIRCUITO'].nunique():,}")
if MODO_AUDITORIA:
    print("Aviso: estas tablas son una muestra para auditar el flujo, no el resultado completo.")
else:
    print("Modo produccion activo: se usaran todos los eventos disponibles.")


Eventos en trabajo : 159,470
Vanos en trabajo   : 27,390
Circuitos trabajo  : 208
Modo produccion activo: se usaran todos los eventos disponibles.


## 3. Modelo y extractor Top-20

Se reutiliza el patron del cuaderno 05: un adaptador `predict_proba` para el clasificador MGCECDL y `KernelShapTopVarsExtractor` para obtener las variables Top-20 por evento. Esta es la etapa mas costosa del flujo porque calcula Kernel SHAP para cada fila de trabajo.

In [5]:
from chec_impacto.interpretability.circuit_analysis import KernelShapTopVarsExtractor
from chec_impacto.training import cargar_modelo_mgcecdl, predict_classification

# Cargar el clasificador entrenado sin ruido de warnings en la salida del cuaderno.
with warnings.catch_warnings(), redirect_stdout(io.StringIO()):
    warnings.simplefilter("ignore")
    modelo = cargar_modelo_mgcecdl(str(MODEL_PATH), device=DEVICE)


# KernelShapTopVarsExtractor espera una interfaz predict_proba estilo sklearn.
class MGCECDLClassifierShapAdapter:
    def __init__(self, model, device):
        self.model = model
        self.device = device

    def predict_proba(self, values):
        values = np.asarray(values, dtype=np.float32)
        if values.ndim == 1:
            values = values.reshape(1, -1)
        return np.asarray(
            predict_classification(self.model, values, device=self.device)["fused_probs"],
            dtype=np.float64,
        )


# El extractor calcula SHAP por lotes y cachea el Top-20 de cada fila.
modelo_shap = MGCECDLClassifierShapAdapter(modelo, DEVICE)
shap_extractor = KernelShapTopVarsExtractor(
    model=modelo_shap,
    X=X,
    features=features,
    top_k=TOP_K_VARS,
    background_size=SHAP_BACKGROUND_SIZE,
    nsamples=SHAP_NSAMPLES,
    batch_size=SHAP_BATCH_SIZE,
    random_state=SHAP_RANDOM_STATE,
)

print(f"Modelo: {type(modelo).__name__} | {MODEL_PATH.name}")
print(
    f"Kernel SHAP configurado | fondo={shap_extractor.n_background} "
    f"| nsamples={SHAP_NSAMPLES} | batch={SHAP_BATCH_SIZE}"
)

Modelo: MGCECDLClassifier | mgcecdl_classifier_best.zip
Kernel SHAP configurado | fondo=40 | nsamples=80 | batch=512


## 4. Documento 1 - Top-20 por evento

Replica la estructura de `Indicadores_vano_v3_top20_inferencia_clasificacion.csv`, usando la columna `TOP_20_VARIABLES_MGCECDL`. El resultado se muestra para control y se guarda como CSV en la carpeta MGCECDL.

In [6]:
# Normalizar escalares numpy/pandas para que el diccionario sea serializable.
def _scalar_python(value):
    if pd.isna(value):
        return None
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).isoformat()
    if isinstance(value, pd.Timedelta):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def construir_top_vars_con_valores(row, top_col="_TOP_VARS"):
    salida = {}
    top_vars = row.get(top_col, {})
    if not isinstance(top_vars, dict):
        return salida
    for variable, relevancia in list(top_vars.items())[:TOP_K_VARS]:
        salida[variable] = {
            "valor_original": _scalar_python(row[variable]) if variable in row.index else None,
            "relevancia_mgcecdl": float(relevancia),
        }
    return salida


from tqdm.auto import tqdm

# Calcular el Top-20 por evento con Kernel SHAP. Esta llamada puede tardar con todos los datos.
# Se procesa por lotes para que tqdm muestre avance, velocidad y tiempo estimado restante.
indices_eventos = np.arange(len(base))
top_vars_por_evento = []
for start in tqdm(
    range(0, len(indices_eventos), SHAP_BATCH_SIZE),
    total=int(np.ceil(len(indices_eventos) / SHAP_BATCH_SIZE)),
    desc="Kernel SHAP Top-20 MGCECDL",
    unit="lote",
):
    batch_indices = indices_eventos[start:start + SHAP_BATCH_SIZE]
    top_vars_por_evento.extend(shap_extractor.calcular_top_vars(batch_indices))

base_top = base.copy()
base_top["_TOP_VARS"] = top_vars_por_evento
tabla_top20_eventos = base_top.drop(columns=["_TOP_VARS"]).copy()
tabla_top20_eventos["TOP_20_VARIABLES_MGCECDL"] = base_top.apply(
    construir_top_vars_con_valores,
    axis=1,
)

print("Documento 1 - tabla_top20_eventos")
print("Shape:", tabla_top20_eventos.shape)
display(tabla_top20_eventos.head(10))
guardar_resultado(tabla_top20_eventos, OUTPUT_FILES["top20_eventos"], "Documento 1 - Top-20 por evento")

Kernel SHAP Top-20 MGCECDL: 100%|██████████| 312/312 [1:19:57<00:00, 15.38s/lote]


Documento 1 - tabla_top20_eventos
Shape: (159470, 274)


,CIRCUITO,FID_SW,COD_EQ_PROTEGE,FID_VANO,T_USUS_EQ_PROT,LVSW,CNT_VN,CNT_VN_SW,FECHA,DURACION,...,clouds_16,clouds_17,clouds_18,clouds_19,clouds_20,clouds_21,clouds_22,clouds_23,clouds_24,TOP_20_VARIABLES_MGCECDL
0,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-03 23:56:04,0.008,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,{'wind_gust_spd_3': {'valor_original': 10.7999...
1,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-18 17:08:14,0.008,...,100.0,84.0,99.0,99.0,98.0,98.0,97.0,98.0,100.0,{'TIPO_TAX': {'valor_original': 'Troncal_ramal...
2,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-19 20:25:50,0.008,...,99.0,100.0,100.0,100.0,97.0,100.0,100.0,99.0,98.0,"{'CNT_TRF': {'valor_original': 145, 'relevanci..."
3,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-10 21:33:41,0.008,...,93.0,100.0,55.0,61.0,32.0,56.0,59.0,58.0,76.0,"{'CNT_TRF': {'valor_original': 145, 'relevanci..."
4,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-19 15:59:10,0.041,...,97.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,{'temp_11': {'valor_original': 12.142999649047...
5,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-19 16:35:24,0.727,...,99.0,97.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,{'temp_7': {'valor_original': 17.1930007934570...
6,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-22 23:21:44,0.008,...,95.0,100.0,87.0,64.0,34.0,42.0,34.0,100.0,98.0,{'prep_6': {'valor_original': 0.30000001192092...
7,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-23 01:38:59,0.008,...,98.0,100.0,95.0,100.0,87.0,64.0,34.0,42.0,34.0,"{'CNT_VN': {'valor_original': 16, 'relevancia_..."
8,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-24 00:53:27,0.008,...,95.0,100.0,88.0,94.0,93.0,98.0,100.0,99.0,97.0,"{'ALTURA': {'valor_original': 10.0, 'relevanci..."
9,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-30 03:17:14,0.008,...,43.0,63.0,95.0,97.0,77.0,97.0,88.0,86.0,88.0,{'TIPO_TAX': {'valor_original': 'Troncal_ramal...


Guardado Documento 1 - Top-20 por evento: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_top20_mgcecdl_clasificacion.csv | filas=159,470 | columnas=274


## 5. Documento 2 - Agregado por vano

Replica la estructura de `Indicadores_vano_v3_agrupado_por_vano.csv`: `FID_VANO`, `CIRCUITO`, apariciones, UITI sumado, Top-20 agregado por Borda ponderado por SHAP y nivel de riesgo. El riesgo se calcula por cuartiles del UITI sumado por vano.

In [7]:
# Borda ponderado resume rankings Top-20 usando posicion y magnitud SHAP.
def agregar_borda_ponderado(df, group_cols, top_col="_TOP_VARS", top_k=20):
    records = []
    for _, row in df.iterrows():
        top_vars = row.get(top_col)
        if not isinstance(top_vars, dict):
            continue
        grupo = {col: row[col] for col in group_cols}
        for pos, (variable, shap_abs) in enumerate(list(top_vars.items())[:top_k], start=1):
            records.append({
                **grupo,
                "_var": variable,
                "_score": float(top_k + 1 - pos) * float(shap_abs),
            })

    if not records:
        return pd.DataFrame(columns=group_cols + ["RELEVANCIA_VARS"])

    exp = pd.DataFrame(records)
    scores = (
        exp.groupby(group_cols + ["_var"], dropna=False, sort=False)["_score"]
        .sum()
        .reset_index()
    )
    scores = scores.sort_values(
        group_cols + ["_score"],
        ascending=[True] * len(group_cols) + [False],
        kind="stable",
    )
    scores["_rank"] = scores.groupby(group_cols, sort=False).cumcount()
    scores = scores[scores["_rank"] < top_k].copy()
    scores["_item"] = list(zip(scores["_var"], scores["_score"]))

    return (
        scores.groupby(group_cols, dropna=False, sort=False)["_item"]
        .agg(lambda items: {var: float(score) for var, score in items})
        .rename("RELEVANCIA_VARS")
        .reset_index()
    )


# El nivel de riesgo se define sobre el UITI sumado por vano.
def asignar_riesgo_vano(tabla_vano):
    out = tabla_vano.copy()
    labels = ["Bajo", "Medio", "Alto", "Muy Alto"]
    if out["UITI_VANO"].nunique(dropna=True) < 4:
        out["RIESGO_VANO"] = "Sin corte"
        return out
    ranking_estable = out["UITI_VANO"].rank(method="first")
    out["RIESGO_VANO"] = pd.qcut(ranking_estable, q=4, labels=labels).astype(str)
    return out


# Agregacion principal por vano: conteo de apariciones y suma de UITI.
metricas_vano = (
    base_top.groupby("FID_VANO", dropna=False, sort=False)
    .agg(
        CIRCUITO=("CIRCUITO", "first"),
        N_APARICIONES=("FID_VANO", "size"),
        UITI_VANO=("UITI_VANO", "sum"),
    )
    .reset_index()
)
# Agregar la relevancia Top-20 de los eventos asociados a cada vano.
relevancia_vano = agregar_borda_ponderado(base_top, ["FID_VANO"], top_k=TOP_K_VARS).rename(
    columns={"RELEVANCIA_VARS": "TOP_20_VARIABLES_VANO"}
)
tabla_vano = metricas_vano.merge(relevancia_vano, on="FID_VANO", how="left")
tabla_vano = asignar_riesgo_vano(tabla_vano)
tabla_vano = tabla_vano[
    ["FID_VANO", "CIRCUITO", "N_APARICIONES", "UITI_VANO", "TOP_20_VARIABLES_VANO", "RIESGO_VANO"]
]

print("Documento 2 - tabla_vano")
print("Shape:", tabla_vano.shape)
display(tabla_vano.head(20))
display(tabla_vano["RIESGO_VANO"].value_counts(dropna=False).rename("N_VANOS"))
guardar_resultado(tabla_vano, OUTPUT_FILES["agrupado_por_vano"], "Documento 2 - Agregado por vano")


Documento 2 - tabla_vano
Shape: (27390, 6)


,FID_VANO,CIRCUITO,N_APARICIONES,UITI_VANO,TOP_20_VARIABLES_VANO,RIESGO_VANO
0,20139439,AGU23L14,20,1602.601298,"{'TIPO_TAX': 24.332644708359744, 'CNT_TRF': 13...",Muy Alto
1,20139477,AGU23L14,20,10.609121,"{'CNT_VN': 115.32486148973076, 'LONG_CRUCETA':...",Bajo
2,20139478,AGU23L14,20,101.098680,"{'CNT_VN': 36.51436249552728, 'X2': 10.2905125...",Medio
3,20139508,AGU23L14,20,36.819890,"{'CNT_VN': 85.35991732275167, 'LONGITUD': 3.23...",Medio
4,20139509,AGU23L14,20,89.241427,"{'CNT_VN': 67.63576784957681, 'Y2': 12.0642989...",Medio
5,20139520,AGU23L14,20,1684.978000,"{'TIPO_TAX': 31.28842649236269, 'CNT_TRF': 20....",Muy Alto
6,20139654,AGU23L14,20,172.242196,"{'PROMEDIO_KWH_VANO': 22.648698389089592, 'CON...",Alto
7,20139655,AGU23L14,20,268.348348,"{'CNT_VN': 47.47539394179181, 'CNT_TRF': 42.13...",Alto
8,20139656,AGU23L14,20,304.544172,"{'CNT_VN': 67.56325957291627, 'Y2': 48.8723682...",Alto
9,20139657,AGU23L14,20,445.583071,"{'CNT_VN': 84.93278175227701, 'Y2': 55.8043153...",Alto


RIESGO_VANO
Muy Alto    6848
Bajo        6848
Medio       6847
Alto        6847
Name: N_VANOS, dtype: int64

Guardado Documento 2 - Agregado por vano: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_agrupado_por_vano.csv | filas=27,390 | columnas=6


## 6. Documento 3 - Agregado por circuito

Replica la estructura de `Indicadores_vano_v3_agrupado_por_circuito.csv`: conteos por circuito, UITI sumado y Top-20 agregado por Borda ponderado por SHAP. El Top-20 del circuito resume los rankings de sus eventos.

In [8]:
# Agregacion por circuito: vanos distintos, apariciones y UITI total.
metricas_circuito = (
    base_top.groupby("CIRCUITO", dropna=False, sort=False)
    .agg(
        N_VANOS=("FID_VANO", "nunique"),
        N_APARICIONES=("FID_VANO", "size"),
        UITI_CIRCUITO=("UITI_VANO", "sum"),
    )
    .reset_index()
)
# Top-20 de circuito a partir de rankings y magnitudes SHAP de todos sus eventos.
relevancia_circuito = agregar_borda_ponderado(base_top, ["CIRCUITO"], top_k=TOP_K_VARS).rename(
    columns={"RELEVANCIA_VARS": "TOP_20_VARIABLES_CIRCUITO"}
)

tabla_circuito = metricas_circuito.merge(relevancia_circuito, on="CIRCUITO", how="left")
tabla_circuito = tabla_circuito[
    ["CIRCUITO", "N_VANOS", "N_APARICIONES", "UITI_CIRCUITO", "TOP_20_VARIABLES_CIRCUITO"]
]

print("Documento 3 - tabla_circuito")
print("Shape:", tabla_circuito.shape)
display(tabla_circuito.head(20))
guardar_resultado(tabla_circuito, OUTPUT_FILES["agrupado_por_circuito"], "Documento 3 - Agregado por circuito")

Documento 3 - tabla_circuito
Shape: (208, 5)


,CIRCUITO,N_VANOS,N_APARICIONES,UITI_CIRCUITO,TOP_20_VARIABLES_CIRCUITO
0,AGU23L14,248,941,104106.054498,"{'TIPO': 1216.7176645564332, 'CNT_VN': 1101.22..."
1,AGU23L15,328,1045,90284.498285,"{'TIPO': 1786.3209497603, 'CNT_TRF': 1218.4523..."
2,AMA23L12,226,2219,324689.834805,"{'Y2': 4620097836145735.0, 'prep_11': 43890929..."
3,AMA23L15,198,1567,92872.353971,"{'CNT_TRF': 1490.170465075031, 'TIPO': 1260.97..."
4,AMA23L16,127,184,97846.122960,"{'TIPO': 281.4790999626862, 'CNT_TRF': 211.625..."
5,AMR23L12,128,615,31430.915819,"{'CNT_TRF': 498.1822933642026, 'TIPO': 356.401..."
6,AMR23L13,362,1497,88398.334126,"{'CNT_TRF': 1217.2874732005646, 'CNT_VN': 1113..."
7,AMR23L14,341,1792,56302.169729,"{'CNT_VN': 2134.8385503629625, 'CNT_TRF': 2126..."
8,AZA23L13,196,572,102440.041179,"{'TIPO': 842.1031165217439, 'CNT_VN': 418.7823..."
9,AZA23L18,173,939,90563.712497,"{'CNT_VN': 394.118515409173, 'CNT_TRF': 292.32..."


Guardado Documento 3 - Agregado por circuito: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_agrupado_por_circuito.csv | filas=208 | columnas=5


## 7. Documento 4 - Resumen por circuito y nivel de criticidad

Replica la estructura de `Indicadores_vano_v3_resumen_circuito_por_nivel_criticidad.csv`. El riesgo se asigna a nivel de vano y luego se propaga a los eventos para calcular porcentajes, UITI sumado y Top-20 por nivel mediante Borda ponderado por SHAP.

In [9]:
# Propagar el riesgo calculado por vano hacia cada evento antes de resumir por nivel.
riesgo_por_vano = tabla_vano[["FID_VANO", "RIESGO_VANO"]].drop_duplicates("FID_VANO")
base_riesgo = base_top.merge(riesgo_por_vano, on="FID_VANO", how="left")

# Totales por circuito usados como denominadores de los porcentajes.
totales_circuito = (
    base_riesgo.groupby("CIRCUITO", dropna=False, sort=False)
    .agg(
        TOTAL_VANOS_CIRCUITO=("FID_VANO", "nunique"),
        TOTAL_APARICIONES_CIRCUITO=("FID_VANO", "size"),
    )
    .reset_index()
)
# Metricas por par circuito-riesgo, usando suma para UITI_NIVEL.
metricas_nivel = (
    base_riesgo.groupby(["CIRCUITO", "RIESGO_VANO"], dropna=False, sort=False)
    .agg(
        N_VANOS_NIVEL=("FID_VANO", "nunique"),
        N_APARICIONES_NIVEL=("FID_VANO", "size"),
        UITI_NIVEL=("UITI_VANO", "sum"),
    )
    .reset_index()
)
# Top-20 por nivel de criticidad dentro de cada circuito, ponderado por SHAP.
relevancia_nivel = agregar_borda_ponderado(
    base_riesgo,
    ["CIRCUITO", "RIESGO_VANO"],
    top_k=TOP_K_VARS,
).rename(columns={"RELEVANCIA_VARS": "TOP_20_VARIABLES_NIVEL"})

tabla_resumen_criticidad = (
    metricas_nivel
    .merge(totales_circuito, on="CIRCUITO", how="left")
    .merge(relevancia_nivel, on=["CIRCUITO", "RIESGO_VANO"], how="left")
)
tabla_resumen_criticidad["PORCENTAJE_VANOS_CIRCUITO"] = (
    tabla_resumen_criticidad["N_VANOS_NIVEL"]
    / tabla_resumen_criticidad["TOTAL_VANOS_CIRCUITO"].replace(0, np.nan)
    * 100
)
tabla_resumen_criticidad["PORCENTAJE_APARICIONES_CIRCUITO"] = (
    tabla_resumen_criticidad["N_APARICIONES_NIVEL"]
    / tabla_resumen_criticidad["TOTAL_APARICIONES_CIRCUITO"].replace(0, np.nan)
    * 100
)
tabla_resumen_criticidad = tabla_resumen_criticidad[
    [
        "CIRCUITO",
        "RIESGO_VANO",
        "N_VANOS_NIVEL",
        "PORCENTAJE_VANOS_CIRCUITO",
        "N_APARICIONES_NIVEL",
        "PORCENTAJE_APARICIONES_CIRCUITO",
        "UITI_NIVEL",
        "TOP_20_VARIABLES_NIVEL",
    ]
]

orden_riesgo = pd.CategoricalDtype(["Bajo", "Medio", "Alto", "Muy Alto", "Sin corte"], ordered=True)
tabla_resumen_criticidad["_ORDEN_RIESGO"] = tabla_resumen_criticidad["RIESGO_VANO"].astype(orden_riesgo)
tabla_resumen_criticidad = (
    tabla_resumen_criticidad
    .sort_values(["CIRCUITO", "_ORDEN_RIESGO"], kind="stable")
    .drop(columns="_ORDEN_RIESGO")
    .reset_index(drop=True)
)

print("Documento 4 - tabla_resumen_criticidad")
print("Shape:", tabla_resumen_criticidad.shape)
display(tabla_resumen_criticidad.head(40))
guardar_resultado(tabla_resumen_criticidad, OUTPUT_FILES["resumen_criticidad"], "Documento 4 - Resumen por criticidad")

Documento 4 - tabla_resumen_criticidad
Shape: (740, 8)


,CIRCUITO,RIESGO_VANO,N_VANOS_NIVEL,PORCENTAJE_VANOS_CIRCUITO,N_APARICIONES_NIVEL,PORCENTAJE_APARICIONES_CIRCUITO,UITI_NIVEL,TOP_20_VARIABLES_NIVEL
0,AGU23L12,Bajo,25,53.191489,26,53.061224,242.541925,"{'Y2': 27.78481388788739, 'CNT_TRF': 16.517636..."
1,AGU23L12,Medio,9,19.148936,10,20.408163,496.229286,"{'TIPO_TAX': 10.242608312156305, 'CNT_TRF': 8...."
2,AGU23L12,Alto,13,27.659574,13,26.530612,4004.379733,"{'TIPO': 49.40094024090483, 'CNT_TRF': 21.0108..."
3,AGU23L13,Bajo,4,12.500000,4,12.121212,59.620503,"{'CNT_VN': 7.028598882156846, 'wind_spd_0': 5...."
4,AGU23L13,Medio,17,53.125000,18,54.545455,1473.203295,"{'TIPO': 18.487637108367313, 'temp_11': 15.078..."
5,AGU23L13,Alto,11,34.375000,11,33.333333,2803.631324,"{'TIPO': 33.63577369564046, 'CNT_TRF': 18.3424..."
6,AGU23L14,Bajo,48,19.354839,91,9.670563,563.588025,"{'CNT_VN': 138.35969915447748, 'TIPO': 96.2591..."
7,AGU23L14,Medio,72,29.032258,176,18.703507,4991.815407,"{'TIPO': 306.6504740218717, 'CNT_VN': 255.1622..."
8,AGU23L14,Alto,60,24.193548,222,23.591923,16349.389460,"{'TIPO': 347.9368677323169, 'CNT_VN': 244.9348..."
9,AGU23L14,Muy Alto,68,27.419355,452,48.034006,82201.261606,"{'CNT_TRF': 481.1233136233256, 'TIPO': 465.871..."


Guardado Documento 4 - Resumen por criticidad: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_resumen_circuito_por_nivel_criticidad.csv | filas=740 | columnas=8


## 8. Control final y manifiesto de salidas

Esta celda valida que los cuatro documentos tengan el esquema esperado y muestra las rutas guardadas. Solo valida forma; los valores dependen del modelo MGCECDL y de los datos procesados.

In [10]:
# Validacion defensiva: confirma que las columnas exportadas sean las esperadas.
columnas_referencia = {
    "top20_eventos": [*base.columns.tolist(), "TOP_20_VARIABLES_MGCECDL"],
    "vano": ["FID_VANO", "CIRCUITO", "N_APARICIONES", "UITI_VANO", "TOP_20_VARIABLES_VANO", "RIESGO_VANO"],
    "circuito": ["CIRCUITO", "N_VANOS", "N_APARICIONES", "UITI_CIRCUITO", "TOP_20_VARIABLES_CIRCUITO"],
    "resumen_criticidad": [
        "CIRCUITO",
        "RIESGO_VANO",
        "N_VANOS_NIVEL",
        "PORCENTAJE_VANOS_CIRCUITO",
        "N_APARICIONES_NIVEL",
        "PORCENTAJE_APARICIONES_CIRCUITO",
        "UITI_NIVEL",
        "TOP_20_VARIABLES_NIVEL",
    ],
}

validacion_esquema = pd.DataFrame(
    [
        {
            "documento": "top20_eventos",
            "columnas_ok": list(tabla_top20_eventos.columns) == columnas_referencia["top20_eventos"],
            "n_columnas": tabla_top20_eventos.shape[1],
            "n_filas": tabla_top20_eventos.shape[0],
        },
        {
            "documento": "agrupado_por_vano",
            "columnas_ok": list(tabla_vano.columns) == columnas_referencia["vano"],
            "n_columnas": tabla_vano.shape[1],
            "n_filas": tabla_vano.shape[0],
        },
        {
            "documento": "agrupado_por_circuito",
            "columnas_ok": list(tabla_circuito.columns) == columnas_referencia["circuito"],
            "n_columnas": tabla_circuito.shape[1],
            "n_filas": tabla_circuito.shape[0],
        },
        {
            "documento": "resumen_circuito_por_nivel_criticidad",
            "columnas_ok": list(tabla_resumen_criticidad.columns) == columnas_referencia["resumen_criticidad"],
            "n_columnas": tabla_resumen_criticidad.shape[1],
            "n_filas": tabla_resumen_criticidad.shape[0],
        },
    ]
)

display(validacion_esquema)
assert validacion_esquema["columnas_ok"].all(), "Hay diferencias de esquema por revisar."
print("Listo: las cuatro tablas fueron mostradas y guardadas.")
for nombre, ruta in OUTPUT_FILES.items():
    print(f"{nombre}: {ruta}")

,documento,columnas_ok,n_columnas,n_filas
0,top20_eventos,True,274,159470
1,agrupado_por_vano,True,6,27390
2,agrupado_por_circuito,True,5,208
3,resumen_circuito_por_nivel_criticidad,True,8,740


Listo: las cuatro tablas fueron mostradas y guardadas.
top20_eventos: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_top20_mgcecdl_clasificacion.csv
agrupado_por_vano: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_agrupado_por_vano.csv
agrupado_por_circuito: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_agrupado_por_circuito.csv
resumen_criticidad: /Users/diego/Documents/CHEC/chec_impacto/data/results_mgcecdl/Indicadores_vano_v3_resumen_circuito_por_nivel_criticidad.csv
